# KWJA で文節・基本句の係り受けと並列構造を解析する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/15_parsing/kwja_dependency.ipynb)

NLP クックブック 15-05 ／ 構文解析

京大コーパスの基準 (KNP 形式) で、文節単位・基本句単位の係り受けを得たい。並列や同格の関係も区別したい。

In [1]:
!pip install -q kwja

In [2]:
import subprocess
from rhoknp import Document

def kwja(text, model_size="base"):
    """KWJA を実行して、解析結果を rhoknp の Document として返す (初回はモデルのダウンロードに数分かかる)"""
    result = subprocess.run(["kwja", "--text", text, "--tasks", "char,word", "--model-size", model_size],
                            capture_output=True, text=True, check=True)
    return Document.from_knp(result.stdout)

doc = kwja("太郎と花子は、昨日京都で買った抹茶とお菓子を先生に渡した。")
DEP = {"D": "係り受け", "P": "並列", "A": "同格", "I": "部分並列"}
print("文節:")
for p in doc.phrases:
    print(f"  {p.text:　<8} → {p.parent.text if p.parent else '(文末)':　<8} {DEP[p.dep_type.value]}")

文節:
  太郎と　　　　　 → 花子は、　　　　 並列
  花子は、　　　　 → 渡した。　　　　 係り受け
  昨日　　　　　　 → 買った　　　　　 係り受け
  京都で　　　　　 → 買った　　　　　 係り受け
  買った　　　　　 → お菓子を　　　　 係り受け
  抹茶と　　　　　 → お菓子を　　　　 並列
  お菓子を　　　　 → 渡した。　　　　 係り受け
  先生に　　　　　 → 渡した。　　　　 係り受け
  渡した。　　　　 → (文末)　　　　 係り受け


In [3]:
print("基本句 (文節をさらに細かくした、係り受けと述語項構造の単位):")
for bp in doc.base_phrases:
    print(f"  {bp.text:　<6} → {bp.parent.text if bp.parent else '(文末)':　<6} {bp.dep_type.value}  主辞: {bp.head.text}")

print([(m.text, m.reading, m.lemma, m.pos, m.subpos) for m in doc.morphemes][:6])

基本句 (文節をさらに細かくした、係り受けと述語項構造の単位):
  太郎と　　　 → 花子は、　　 P  主辞: 太郎
  花子は、　　 → 渡した。　　 D  主辞: 花子
  昨日　　　　 → 買った　　　 D  主辞: 昨日
  京都で　　　 → 買った　　　 D  主辞: 京都
  買った　　　 → お菓子を　　 D  主辞: 買った
  抹茶と　　　 → お菓子を　　 P  主辞: 抹茶
  お菓子を　　 → 渡した。　　 D  主辞: 菓子
  先生に　　　 → 渡した。　　 D  主辞: 先生
  渡した。　　 → (文末)　　 D  主辞: 渡した
[('太郎', 'たろう', '太郎', '名詞', '人名'), ('と', 'と', 'と', '助詞', '格助詞'), ('花子', 'はなこ', '花子', '名詞', '人名'), ('は', 'は', 'は', '助詞', '副助詞'), ('、', '、', '、', '特殊', '読点'), ('昨日', 'きのう', '昨日', '名詞', '時相名詞')]


- 「太郎と → 花子は」「抹茶と → お菓子を」は、普通の係り受け (D) ではなく並列 (P) として区別される。UD (GiNZA) では並列も `nmod` になってしまうので、並列構造を扱いたいときはこちらが便利。
- 解析結果は KNP 形式。`rhoknp` で文 → 文節 → 基本句 → 形態素の階層をたどれる。
- 形態素は JUMAN の品詞体系・辞書に基づく。MeCab + UniDic とは単語の区切りも品詞名も違う点に注意。
- インストール時に Transformers のバージョンが入れ替わることがある。import でエラーになったらランタイムを再起動する。